# Added value of liberal OFFs for predicting NREM delta power: 48h morphological

Do the OFFs that only the liberal criteria keep, CLAS-exclusive and especially
LLAS-exclusive (the small/short/narrow events too small for the conservative BLAS set),
carry unique information about NREM delta power after accounting for what the
conservative (BLAS) OFFs already explain?

"Added value" is an incremental, conditional quantity, not a comparison of marginal
correlations computed separately on nested subsets. Everything below follows from that.

## Design

- Unit: fixed-length time epochs within NREM, per `(subject, probe, structure)`.
- Outcome: epoch-mean `log10(delta)` over all NREM samples in the epoch, never
  restricted to OFF intervals. This breaks the mechanical coupling that inflated the
  per-OFF "total in-OFF bandpower" used elsewhere.
- Predictors, one per disjoint tier: total OFF area in the epoch from `blas_area`
  (BLAS), `clas_excl_area` (CLAS-but-not-BLAS), `llas_excl_area` (LLAS-but-not-CLAS).
  Disjoint tiers minimise collinearity and make each coefficient mean the unique
  delta-relevance of that tier's events.
- Model, per group, all variables z-scored within group:
  `z(mean_log_delta) ~ z(blas_area) + z(clas_excl_area) + z(llas_excl_area)`. The
  headline is the standardized partial coefficient of `llas_excl_area`: the added value
  of the liberal-only OFFs, holding BLAS and CLAS fixed.
- Inference: per-group OLS with HAC (Newey-West) standard errors. Delta drifts on a
  minutes timescale, so epochs are strongly autocorrelated and naive `1/(n-3)` variances
  are anticonservative. Per-group signed coefficients are pooled across groups by
  DerSimonian-Laird random-effects meta-analysis.
- Picture: within BLAS-area strata, mean delta against LLAS-exclusive-area quantile, a
  model-light view of added value as conditional association after holding BLAS fixed.

## What this avoids

- Marginal-correlation comparison across nested subsets (non-incremental,
  range-restriction confounded).
- An outcome defined over OFF intervals only (mechanical coupling).
- `1/(n-3)` variance on autocorrelated epochs (pseudoreplication).
- A `sign(one coef) * sqrt(dR2)` effect size (non-negative magnitude with an invented
  sign that only averages to ~0 by cancellation).
- Single-subject anecdote; all cortical groups are meta-analyzed.

Source: full-48h morphological cortical OFFs, the current best-but-provisional source.
The categorized OFF frame already cached by `static_added_value.ipynb` (`direct_48h`
mode) is reused; the delta bandpower series is loaded per group fresh.


In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats
import statsmodels.api as sm
import xarray as xr
from statsmodels.stats.outliers_influence import variance_inflation_factor

from cnpix_local_sleep import files, hyp

In [ ]:
# ---- config ----
# Reuse the categorized 48h morphological cortical OFFs built by static_added_value.ipynb.
CACHE_PARQUET = pathlib.Path(
    "./outputs/static_added_value/cache/offs_direct_48h.parquet"
)
OUTPUT_DIR = pathlib.Path("./outputs/incremental_added_value")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
save_plots = True

GROUP_COLS = ["subject", "probe", "structure"]

# Disjoint tier -> epoch predictor. In the cached frame, category=="BLAS" is BLAS;
# "CLAS" is CLAS-but-not-BLAS (CLAS-exclusive); "LLAS" is LLAS-but-not-CLAS
# (LLAS-exclusive, the liberal-only events).
TIER_TO_FEATURE = {
    "BLAS": "blas_area",
    "CLAS": "clas_excl_area",
    "LLAS": "llas_excl_area",
}
PREDICTORS = ["blas_area", "clas_excl_area", "llas_excl_area"]
HEADLINE = "llas_excl_area"  # added value of the liberal-only OFFs

EPOCH_DURATION = 10.0  # s (primary)
MIN_NREM_FRAC = 0.8    # keep an epoch only if >=80% of its samples are clean NREM
MIN_EPOCHS = 50        # per-group minimum
HAC_TARGET_S = 300.0   # Newey-West maxlags chosen to cover ~5 min of autocorrelation
EPOCH_SWEEP = [4.0, 10.0, 30.0]  # sensitivity
RUN_BOOTSTRAP = False  # set True for the (slower) moving-block bootstrap robustness check

## Load categorized NREM OFFs

In [ ]:
offs = pd.read_parquet(
    CACHE_PARQUET,
    columns=GROUP_COLS + ["start_time", "area", "span", "category", "state"],
)
offs = offs[offs["state"] == "NREM"].reset_index(drop=True)
n_groups = offs.groupby(GROUP_COLS, observed=True).ngroups
print(f"{len(offs):,} NREM cortical OFFs across {n_groups} groups")
print(offs["category"].value_counts().to_dict())

## Per-group delta loader (NREM, finite, log10)

The outcome is delta over all clean-NREM samples in a window, never restricted to OFF
intervals, so the OFF-area predictors cannot be definitionally coupled to it.


In [ ]:
_DELTA_CACHE = {}


def load_group_delta_nrem(subject, probe, structure):
    """Return (time, log10_delta, fs) for clean-NREM, finite delta samples."""
    key = (subject, probe, structure)
    if key in _DELTA_CACHE:
        return _DELTA_CACHE[key]
    da = xr.load_dataarray(
        files.get_structure_bandpower_path(
            subject, probe, structure, "delta", True, "inst"
        )
    )
    t = da["time"].values
    v = da.values
    hg = hyp.load_statistical_condition_hypnograms(subject, probe)["Full.Conservative"]
    nrem = hg.keep_states(["NREM"]).covers_time(t)
    keep = nrem & np.isfinite(v)
    fs = 1.0 / np.median(np.diff(t[:10000]))  # underlying sampling rate
    _DELTA_CACHE[key] = (t[keep], np.log10(v[keep]), fs)
    return _DELTA_CACHE[key]

## Epoch table builder

In [ ]:
def build_epoch_table(group_offs, t, logd, fs, epoch_duration, min_nrem_frac):
    """Bin clean-NREM time into fixed epochs.

    Per epoch: mean log-delta (over all NREM samples) and total OFF area per
    disjoint tier. Epochs are kept only if >= ``min_nrem_frac`` of their sample
    slots carry clean-NREM delta, so partial NREM/Wake-boundary epochs are dropped.
    """
    if t.size == 0:
        return pd.DataFrame()
    edges = np.arange(t[0], t[-1] + epoch_duration, epoch_duration)
    n = len(edges) - 1
    if n < 1:
        return pd.DataFrame()

    # Outcome: mean log-delta per epoch over clean-NREM samples.
    s_ep = np.clip(np.searchsorted(edges, t, side="right") - 1, 0, n - 1)
    dsum = np.zeros(n)
    dcnt = np.zeros(n)
    np.add.at(dsum, s_ep, logd)
    np.add.at(dcnt, s_ep, 1)
    valid = dcnt >= (min_nrem_frac * epoch_duration * fs)
    mean_log_delta = np.full(n, np.nan)
    mean_log_delta[valid] = dsum[valid] / dcnt[valid]

    # Predictors: total OFF area per disjoint tier per epoch.
    cols = {}
    for tier, feat in TIER_TO_FEATURE.items():
        sub = group_offs[group_offs["category"] == tier]
        area = np.zeros(n)
        if len(sub):
            oe = np.searchsorted(edges, sub["start_time"].to_numpy(), side="right") - 1
            inr = (oe >= 0) & (oe < n)
            np.add.at(area, oe[inr], sub["area"].to_numpy()[inr].astype(float))
        cols[feat] = area

    out = pd.DataFrame(cols)
    out["mean_log_delta"] = mean_log_delta
    out["epoch_start"] = edges[:-1]
    return out.loc[valid].reset_index(drop=True)

## Per-group incremental regression with HAC standard errors

In [ ]:
def zscore(s):
    sd = s.std(ddof=0)
    return (s - s.mean()) / sd if sd > 0 else s * 0.0


def prep_columns(df, transform):
    """Standardize each column. transform='zscore' uses the raw values (the OLS
    model); transform='rank' rank-transforms first (rank-transform regression /
    partial-Spearman analog) -- robust to heavy tails and invariant to monotone
    transforms -- then standardizes the ranks for comparable coefficients."""
    out = df.copy()
    for c in out.columns:
        col = out[c]
        if transform == "rank":
            col = pd.Series(scipy.stats.rankdata(col), index=col.index)
        out[c] = zscore(col)
    return out


def semipartial_scales(df, predictors):
    """sqrt(1 - R2_i) for each predictor, where R2_i is from regressing predictor i
    on the remaining predictors. Multiplying a joint coefficient by this factor gives
    the semipartial (part) correlation: with a standardized outcome it equals
    cor(y, resid_i), and its square is predictor i's incremental R2. The factor is a
    function of the design matrix alone, so the HAC standard error scales by exactly
    the same amount -- no bootstrap or delta method is needed. With one predictor the
    factor is 1, so a marginal coefficient is its own semipartial."""
    out = {}
    for p in predictors:
        others = [q for q in predictors if q != p]
        r2 = (sm.OLS(df[p], sm.add_constant(df[others])).fit().rsquared
              if others else 0.0)
        out[p] = np.sqrt(max(0.0, 1.0 - r2))
    return out


def fit_group(epoch_df, epoch_duration, hac_target_s=HAC_TARGET_S, transform="zscore"):
    """Per-group HAC-OLS of the (z-scored or rank-transformed) tiers on delta.

    transform='zscore' is the primary OLS model; transform='rank' is the rank-based
    sibling (each tier coefficient is a partial-Spearman-flavored association).
    Returns None if too few epochs or any predictor is constant.
    """
    base = epoch_df[PREDICTORS + ["mean_log_delta"]].dropna()
    if len(base) < MIN_EPOCHS:
        return None
    df = prep_columns(base, transform)
    if (df[PREDICTORS].std(ddof=0) == 0).any():
        return None

    y = df["mean_log_delta"]
    X = sm.add_constant(df[PREDICTORS])
    maxlags = max(1, int(np.ceil(hac_target_s / epoch_duration)))
    full = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})

    reduced_cols = [c for c in PREDICTORS if c != HEADLINE]
    red = sm.OLS(y, sm.add_constant(df[reduced_cols])).fit()

    vifs = {PREDICTORS[i]: variance_inflation_factor(X.values, i + 1)
            for i in range(len(PREDICTORS))}
    scales = semipartial_scales(df, PREDICTORS)
    out = {
        "n_epochs": len(df),
        "maxlags": maxlags,
        "beta_blas": full.params["blas_area"],
        "se_blas": full.bse["blas_area"],
        "p_blas": full.pvalues["blas_area"],
        "beta_clas_excl": full.params["clas_excl_area"],
        "se_clas_excl": full.bse["clas_excl_area"],
        "p_clas_excl": full.pvalues["clas_excl_area"],
        "beta_llas_excl": full.params[HEADLINE],
        "se_llas_excl": full.bse[HEADLINE],
        "p_llas_excl": full.pvalues[HEADLINE],
        "R2_full": full.rsquared,
        "delta_R2_llas_excl": full.rsquared - red.rsquared,
        "vif_llas_excl": vifs["llas_excl_area"],
        "vif_max": max(vifs.values()),
        "df_z": df,  # kept for the pooled stratified picture (OLS pass only)
    }
    # Semipartial (part) coefficients -- the reported partial quantity. beta_sr**2 is
    # the tier's incremental R2; beta**2 (the joint coefficient) is not a variance share.
    for pred, col in {"blas_area": "blas", "clas_excl_area": "clas_excl",
                      "llas_excl_area": "llas_excl"}.items():
        out[f"beta_{col}_sr"] = out[f"beta_{col}"] * scales[pred]
        out[f"se_{col}_sr"] = out[f"se_{col}"] * scales[pred]
        out[f"sr_scale_{col}"] = scales[pred]
    return out

In [ ]:
records = []
pooled_parts = []

for keys, grp in offs.groupby(GROUP_COLS, observed=True):
    subject, probe, structure = keys
    t, logd, fs = load_group_delta_nrem(subject, probe, structure)
    et = build_epoch_table(grp, t, logd, fs, EPOCH_DURATION, MIN_NREM_FRAC)
    res = fit_group(et, EPOCH_DURATION)
    if res is None:
        print(f"  skip {keys}: too few/degenerate epochs ({len(et)})")
        continue
    pooled_parts.append(res.pop("df_z").assign(**dict(zip(GROUP_COLS, keys))))
    res.update(dict(zip(GROUP_COLS, keys)))
    records.append(res)

group_df = pd.DataFrame(records)
pooled = pd.concat(pooled_parts, ignore_index=True)
print(f"fitted {len(group_df)} groups; {len(pooled):,} pooled epochs")
print(f"median epochs/group = {group_df['n_epochs'].median():,.0f}; "
      f"max VIF across groups = {group_df['vif_max'].max():.2f}")
display(
    group_df[GROUP_COLS + ["n_epochs", "beta_blas", "beta_clas_excl",
                           "beta_llas_excl", "se_llas_excl", "p_llas_excl",
                           "delta_R2_llas_excl", "vif_llas_excl"]]
    .style.format({
        "beta_blas": "{:+.3f}", "beta_clas_excl": "{:+.3f}",
        "beta_llas_excl": "{:+.3f}", "se_llas_excl": "{:.3f}",
        "p_llas_excl": "{:.2e}", "delta_R2_llas_excl": "{:.4f}",
        "vif_llas_excl": "{:.2f}", "n_epochs": "{:,}",
    })
    .set_caption("Per-group standardized partial coefficients (HAC SEs)")
)

## Random-effects pooling

DerSimonian-Laird on the signed standardized partial coefficient with its HAC variance.
No sign is invented and no Fisher back-transform is applied; the effect is already a
coherent signed quantity.


In [ ]:
def random_effects_meta(effects, variances):
    """DerSimonian-Laird random-effects pooling of generic (effect, variance)."""
    eff = np.asarray(effects, float)
    v = np.asarray(variances, float)
    w = 1.0 / v
    fe = np.sum(w * eff) / np.sum(w)
    q = np.sum(w * (eff - fe) ** 2)
    k = len(eff)
    c = np.sum(w) - np.sum(w**2) / np.sum(w)
    tau2 = max(0.0, (q - (k - 1)) / c) if c > 0 else 0.0
    wre = 1.0 / (v + tau2)
    pooled = np.sum(wre * eff) / np.sum(wre)
    se = 1.0 / np.sqrt(np.sum(wre))
    i2 = max(0.0, (q - (k - 1)) / q) * 100 if q > 0 else 0.0
    zt = pooled / se
    p = 2 * (1 - scipy.stats.norm.cdf(abs(zt)))
    return dict(pooled=pooled, se=se, ci_lo=pooled - 1.96 * se,
               ci_hi=pooled + 1.96 * se, p=p, tau2=tau2, i_squared=i2, k=k)


# Each disjoint tier's partial coefficient (controls for the other two), ordered
# along the OFF size axis big -> small.
TIER_COEFS = {
    "BLAS": ("beta_blas", "se_blas"),
    "CLAS-exclusive": ("beta_clas_excl", "se_clas_excl"),
    "LLAS-exclusive": ("beta_llas_excl", "se_llas_excl"),
}


def pool_tiers(gdf, tiers=("BLAS", "CLAS-exclusive", "LLAS-exclusive"), suffix=""):
    """Random-effects pool each tier's per-group (beta, se) into a tidy table.

    suffix="_sr" pools the semipartial coefficients instead of the joint ones."""
    rows = []
    for name in tiers:
        b, s = TIER_COEFS[name]
        b, s = b + suffix, s + suffix
        m = random_effects_meta(gdf[b], gdf[s] ** 2)
        rows.append(dict(tier=name, pooled_std_beta=m["pooled"], ci_lo=m["ci_lo"],
                         ci_hi=m["ci_hi"], p=m["p"], i_squared=m["i_squared"], k=m["k"]))
    return pd.DataFrame(rows)


tier_meta = {name: random_effects_meta(group_df[b], group_df[s] ** 2)
             for name, (b, s) in TIER_COEFS.items()}
meta_blas = tier_meta["BLAS"]
meta_clas = tier_meta["CLAS-exclusive"]
meta_llas = tier_meta["LLAS-exclusive"]  # headline

meta_summary = pool_tiers(group_df)
# Semipartial (part) coefficients -- the reported partial quantity (see Interpretation).
meta_semipartial = pool_tiers(group_df, suffix="_sr")
for _, r in meta_summary.iterrows():
    tag = "  <- headline" if r["tier"] == "LLAS-exclusive" else ""
    print(f"{r['tier']:16s}: pooled std.beta = {r['pooled_std_beta']:+.4f} "
          f"[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]  p = {r['p']:.2e}  "
          f"I2 = {r['i_squared']:.0f}%  k = {int(r['k'])}{tag}")

display(
    meta_summary.style.format({
        "pooled_std_beta": "{:+.4f}", "ci_lo": "{:+.4f}", "ci_hi": "{:+.4f}",
        "p": "{:.2e}", "i_squared": "{:.0f}%",
    }).set_caption(
        "Pooled standardized partial coefficient by disjoint tier "
        "(each controls for the other two; HAC SEs, DL random-effects)"
    )
)

## Forest plots: added value by tier (LLAS-exclusive vs CLAS-exclusive)

Two panels share the same group ordering (by the headline LLAS-exclusive coefficient)
and the same x-scale, so rows are directly comparable: same y, same group. Each point is
a group's standardized partial coefficient (that tier's OFF area holding the other two
tiers fixed) with its 95% HAC CI; the diamond is the random-effects pooled estimate.


In [ ]:
def forest(group_df, meta, effect_col, se_col, title, ax=None,
           order=None, show_labels=True):
    labels_all = [" / ".join(str(r[c]) for c in GROUP_COLS) for _, r in group_df.iterrows()]
    if order is None:
        order = np.argsort(group_df[effect_col].values)
    gd = group_df.iloc[order].reset_index(drop=True)
    labels = [labels_all[i] for i in order]
    k = len(gd)
    if ax is None:
        _, ax = plt.subplots(figsize=(6, max(4, 0.28 * (k + 3))), constrained_layout=True)
    y = np.arange(k)[::-1]
    eff = gd[effect_col].values
    se = gd[se_col].values
    ax.errorbar(eff, y, xerr=1.96 * se, fmt="o", color="steelblue",
                ecolor="steelblue", elinewidth=1.1, markersize=4, capsize=2)
    yo = -1.5
    hw = 0.45
    ax.fill([meta["ci_lo"], meta["pooled"], meta["ci_hi"], meta["pooled"]],
            [yo, yo + hw, yo, yo - hw], color="firebrick", alpha=0.75)
    ax.axvline(0, color="grey", ls="--", lw=0.8)
    if show_labels:
        ax.set_yticks(list(y) + [yo])
        ax.set_yticklabels(labels + ["RE pooled"], fontsize=7)
    else:
        ax.set_yticks([])
    ax.set_xlabel("standardized partial coefficient")
    ax.set_title(title, fontsize=9)
    ax.set_ylim(yo - 1, y[0] + 1)
    return ax


# Shared ordering (by the headline coefficient) and shared x-scale for comparability.
order = np.argsort(group_df["beta_llas_excl"].values)
lo = min(group_df[["beta_llas_excl", "beta_clas_excl"]].min().min(),
         meta_llas["ci_lo"], meta_clas["ci_lo"]) - 0.05
hi = max(group_df[["beta_llas_excl", "beta_clas_excl"]].max().max(),
         meta_llas["ci_hi"], meta_clas["ci_hi"]) + 0.05

fig, axes = plt.subplots(
    1, 2, figsize=(12, max(4, 0.28 * (len(group_df) + 3))),
    constrained_layout=True, sharex=True,
)
forest(group_df, meta_llas, "beta_llas_excl", "se_llas_excl",
       f"LLAS-exclusive | BLAS, CLAS fixed\npooled {meta_llas['pooled']:+.3f} "
       f"[{meta_llas['ci_lo']:+.3f}, {meta_llas['ci_hi']:+.3f}]",
       ax=axes[0], order=order, show_labels=True)
forest(group_df, meta_clas, "beta_clas_excl", "se_clas_excl",
       f"CLAS-exclusive | BLAS, LLAS-excl fixed\npooled {meta_clas['pooled']:+.3f} "
       f"[{meta_clas['ci_lo']:+.3f}, {meta_clas['ci_hi']:+.3f}]",
       ax=axes[1], order=order, show_labels=False)
for a in axes:
    a.set_xlim(lo, hi)
fig.suptitle("Added value by tier (same group order, same x-scale)", fontsize=10)
if save_plots:
    fig.savefig(OUTPUT_DIR / "forest_added_value_by_tier.svg")
plt.show()

## Marginal vs partial: have value vs add value

Each tier's coefficient alone (marginal: does that tier's OFF area predict delta at
all?) next to its partial coefficient from the joint model (add value: beyond the other
two tiers). This is the have-vs-add contrast that motivated the analysis. A tier can
have a clearly nonzero marginal association yet a near-zero or opposite-signed partial
coefficient once the other tiers are held fixed.


In [ ]:
# Single-predictor (marginal) fit per tier, reusing the cached delta + pooling.
TIER_AREA = {"BLAS": "blas_area",
             "CLAS-exclusive": "clas_excl_area",
             "LLAS-exclusive": "llas_excl_area"}
TIER_COL = {"BLAS": ("beta_blas", "se_blas"),
            "CLAS-exclusive": ("beta_clas_excl", "se_clas_excl"),
            "LLAS-exclusive": ("beta_llas_excl", "se_llas_excl")}

marg_recs = []
maxlags = max(1, int(np.ceil(HAC_TARGET_S / EPOCH_DURATION)))
for keys, grp in offs.groupby(GROUP_COLS, observed=True):
    t, logd, fs = load_group_delta_nrem(*keys)
    et = build_epoch_table(grp, t, logd, fs, EPOCH_DURATION, MIN_NREM_FRAC)
    base = et[PREDICTORS + ["mean_log_delta"]].dropna()
    if len(base) < MIN_EPOCHS:
        continue
    y = zscore(base["mean_log_delta"])
    rec = dict(zip(GROUP_COLS, keys))
    for tier, area in TIER_AREA.items():
        x = zscore(base[area])
        bcol, scol = TIER_COL[tier]
        if x.std(ddof=0) == 0:
            rec[bcol] = np.nan
            rec[scol] = np.nan
        else:
            f = sm.OLS(y, sm.add_constant(x)).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
            rec[bcol] = f.params[area]
            rec[scol] = f.bse[area]
    marg_recs.append(rec)

group_df_marg = pd.DataFrame(marg_recs)
meta_marg = pool_tiers(group_df_marg)

combined_hv = pd.concat([
    meta_marg.assign(kind="marginal (have value)"),
    meta_summary.assign(kind="partial (add value)"),
], ignore_index=True)
for _, r in combined_hv.sort_values(["tier", "kind"]).iterrows():
    print(f"{r['kind']:24s} {r['tier']:16s}: std.beta = {r['pooled_std_beta']:+.4f} "
          f"[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]  p = {r['p']:.2e}")
combined_hv["beta [95% CI]"] = combined_hv.apply(
    lambda r: f"{r['pooled_std_beta']:+.3f} [{r['ci_lo']:+.3f}, {r['ci_hi']:+.3f}]", axis=1
)
display(
    combined_hv.pivot(index="tier", columns="kind", values="beta [95% CI]")
    .style.set_caption("Marginal vs partial standardized coefficient by tier "
                       "(have value vs add value)")
)

## Rank-based robustness sibling (partial Spearman / rank-transform regression)

Per-epoch tier areas are zero-inflated and heavy-tailed, so the linear OLS slope could
be leverage-driven, or a poor single-slope summary. The same incremental model is refit
with every variable rank-transformed within group, then standardized: each tier's
ranked-area coefficient, holding the other ranked tiers fixed, is a rank-based partial
association, a partial-Spearman analog. It is robust to heavy tails and invariant to
monotone transforms, and keeps the HAC SEs, the random-effects pooling and the
conditional framing. A marginal Spearman would throw away the conditioning. Ranks tame
the tails but only partly address zero-inflation: the zero mass collapses into one tie
block. If the size-axis gradient and the signs survive, the heavy-tail and
zero-inflation objection is largely answered.


In [ ]:
# Rank-transform-regression pass (reuses the cached delta series).
rank_recs = []
for keys, grp in offs.groupby(GROUP_COLS, observed=True):
    t, logd, fs = load_group_delta_nrem(*keys)
    et = build_epoch_table(grp, t, logd, fs, EPOCH_DURATION, MIN_NREM_FRAC)
    res = fit_group(et, EPOCH_DURATION, transform="rank")
    if res is not None:
        res.pop("df_z", None)
        res.update(dict(zip(GROUP_COLS, keys)))
        rank_recs.append(res)
group_df_rank = pd.DataFrame(rank_recs)
meta_summary_rank = pool_tiers(group_df_rank)


def methods_wide(long_df, index_cols):
    """Pivot a tidy pooled frame (with a 'method' column) to OLS | rank columns."""
    d = long_df.copy()
    d["beta [95% CI]"] = d.apply(
        lambda r: f"{r['pooled_std_beta']:+.3f} "
                  f"[{r['ci_lo']:+.3f}, {r['ci_hi']:+.3f}]",
        axis=1,
    )
    return d.pivot(index=index_cols, columns="method", values="beta [95% CI]")


combined = pd.concat([
    meta_summary.assign(method="OLS"),
    meta_summary_rank.assign(method="rank"),
], ignore_index=True)
for _, r in combined.sort_values(["tier", "method"]).iterrows():
    print(f"{r['method']:4s} {r['tier']:16s}: pooled std.beta = {r['pooled_std_beta']:+.4f} "
          f"[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]  p = {r['p']:.2e}  I2 = {r['i_squared']:.0f}%")

display(
    methods_wide(combined, ["tier"]).style.set_caption(
        f"Pooled standardized partial coefficient by tier: OLS vs rank-transform "
        f"(primary epoch = {EPOCH_DURATION:.0f}s)"
    )
)

In [ ]:
# Per-group agreement: OLS vs rank standardized coefficient (one point per group x tier).
merged = group_df.merge(group_df_rank, on=GROUP_COLS, suffixes=("_ols", "_rank"))
colors = {"BLAS": "#4c72b0", "CLAS-exclusive": "#dd8452", "LLAS-exclusive": "#55a868"}

fig, ax = plt.subplots(figsize=(5, 5), constrained_layout=True)
for tier, (b, _s) in TIER_COEFS.items():
    ax.scatter(merged[f"{b}_ols"], merged[f"{b}_rank"], s=22, alpha=0.75,
               color=colors[tier], label=tier)
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
        max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, "k--", lw=0.8, label="identity")
ax.axhline(0, color="grey", lw=0.6)
ax.axvline(0, color="grey", lw=0.6)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("OLS standardized partial coef")
ax.set_ylabel("rank-transform standardized partial coef")
ax.set_title("Per-group agreement: OLS vs rank (each point = group x tier)", fontsize=9)
ax.legend(fontsize=7)
if save_plots:
    fig.savefig(OUTPUT_DIR / "ols_vs_rank_agreement.svg")
plt.show()

## Model-light cross-check: does delta track each liberal tier's OFF area within BLAS strata?

Pooled across groups, each group's epochs z-scored first so they are comparable. Within
each BLAS-area stratum, plot mean delta against the focus tier's OFF-area quantile. If a
tier adds value its lines slope away from flat; the two exclusive tiers are expected to
slope in opposite directions, matching their opposite-signed partial coefficients.
Because the x-axis is binned by within-stratum rank quantiles, this picture is already
insensitive to the predictor's heavy tails. It stratifies on BLAS only, whereas the
regression also holds the third tier fixed.


In [ ]:
def stratified_curve(pc, focus_col, n_blas=3, n_q=5):
    d = pc.copy()
    d["blas_stratum"] = pd.qcut(
        d["blas_area"].rank(method="first"), n_blas,
        labels=["BLAS low", "BLAS mid", "BLAS high"],
    )
    d["q"] = d.groupby("blas_stratum", observed=True)[focus_col].transform(
        lambda s: pd.qcut(s.rank(method="first"), n_q, labels=False)
    )
    return (
        d.groupby(["blas_stratum", "q"], observed=True)["mean_log_delta"]
        .agg(["mean", "sem"]).reset_index()
    )


n_q = 5
focus = [("llas_excl_area", "LLAS-exclusive"), ("clas_excl_area", "CLAS-exclusive")]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True, constrained_layout=True)
for ax, (col, label) in zip(axes, focus):
    agg = stratified_curve(pooled, col, n_q=n_q)
    for stratum, sub in agg.groupby("blas_stratum", observed=True):
        ax.errorbar(sub["q"], sub["mean"], yerr=sub["sem"], marker="o",
                    capsize=2, label=str(stratum))
    ax.set_xlabel(f"{label} OFF area (within-stratum quantile)")
    ax.set_title(f"Delta vs {label} OFF area, stratified by BLAS", fontsize=9)
    ax.set_xticks(range(n_q))
axes[0].set_ylabel("mean z(log delta)")
axes[0].legend(title="BLAS stratum", fontsize=8)
if save_plots:
    fig.savefig(OUTPUT_DIR / "stratified_added_value_picture.svg")
plt.show()

## Robustness 1: epoch-length sensitivity

Re-pool each tier's coefficient at several epoch lengths, for both the OLS and
rank-transform models, reusing the cached delta series. Neither the added-value verdict
nor the size-axis sign gradient should hinge on the epoch length or on the
distributional assumptions.


In [ ]:
sweep_rows = []
for ep in EPOCH_SWEEP:
    recs = {"OLS": [], "rank": []}
    for keys, grp in offs.groupby(GROUP_COLS, observed=True):
        t, logd, fs = load_group_delta_nrem(*keys)
        et = build_epoch_table(grp, t, logd, fs, ep, MIN_NREM_FRAC)
        for method, transform in [("OLS", "zscore"), ("rank", "rank")]:
            res = fit_group(et, ep, transform=transform)
            if res is not None:
                res.pop("df_z", None)
                recs[method].append(res)
    for method in ("OLS", "rank"):
        pt = pool_tiers(pd.DataFrame(recs[method]))
        pt.insert(0, "method", method)
        pt.insert(0, "epoch_s", ep)
        sweep_rows.append(pt)

sweep_df = pd.concat(sweep_rows, ignore_index=True)
display(
    methods_wide(sweep_df, ["epoch_s", "tier"]).style.set_caption(
        "Pooled coefficient by tier and epoch length: OLS vs rank-transform"
    )
)

## Robustness 2: first differences

Differencing consecutive epochs removes the slow shared trend (sleep-cycle drift),
asking whether changes in each tier's OFF area track changes in delta beyond the other
tiers' changes, reported for both the OLS and rank-transform models. A short HAC window
guards the residual autocorrelation differencing leaves behind. The exclusive tiers'
signs can differ between levels and differences, which is a timescale effect.


In [ ]:
diff_rows = {"OLS": [], "rank": []}
for keys, grp in offs.groupby(GROUP_COLS, observed=True):
    t, logd, fs = load_group_delta_nrem(*keys)
    et = build_epoch_table(grp, t, logd, fs, EPOCH_DURATION, MIN_NREM_FRAC)
    base = et[PREDICTORS + ["mean_log_delta"]].dropna()
    if len(base) < MIN_EPOCHS + 1:
        continue
    d0 = base.diff().dropna()
    for method, transform in [("OLS", "zscore"), ("rank", "rank")]:
        d = prep_columns(d0, transform)
        if (d[PREDICTORS].std(ddof=0) == 0).any():
            continue
        fit = sm.OLS(d["mean_log_delta"], sm.add_constant(d[PREDICTORS])).fit(
            cov_type="HAC", cov_kwds={"maxlags": 5}
        )
        diff_rows[method].append(dict(
            zip(GROUP_COLS, keys),
            beta_blas=fit.params["blas_area"], se_blas=fit.bse["blas_area"],
            beta_clas_excl=fit.params["clas_excl_area"], se_clas_excl=fit.bse["clas_excl_area"],
            beta_llas_excl=fit.params[HEADLINE], se_llas_excl=fit.bse[HEADLINE],
        ))

diff_long = []
for method in ("OLS", "rank"):
    pt = pool_tiers(pd.DataFrame(diff_rows[method]))
    pt.insert(0, "method", method)
    diff_long.append(pt)
diff_df = pd.concat(diff_long, ignore_index=True)
for _, r in diff_df.sort_values(["tier", "method"]).iterrows():
    print(f"first-diff {r['method']:4s} {r['tier']:16s}: pooled std.beta = "
          f"{r['pooled_std_beta']:+.4f} [{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]  "
          f"p = {r['p']:.2e}")
display(
    methods_wide(diff_df, ["tier"]).style.set_caption(
        "First-differenced pooled coefficient by tier: OLS vs rank-transform"
    )
)

## Robustness 3: moving-block bootstrap (optional)

A non-parametric alternative to HAC: resample contiguous ~5-min blocks of epochs
per group, refit, and use the bootstrap spread as each tier's per-group SE before
pooling. Disabled by default (`RUN_BOOTSTRAP`).

In [ ]:
def block_bootstrap_betas(df_z, epoch_duration, n_boot=300, block_s=300.0, seed=0):
    """Moving-block bootstrap of every tier's partial coefficient."""
    rng = np.random.default_rng(seed)
    bl = max(1, int(round(block_s / epoch_duration)))
    y = df_z["mean_log_delta"].to_numpy()
    X = sm.add_constant(df_z[PREDICTORS]).to_numpy()
    n = len(y)
    n_blocks = int(np.ceil(n / bl))
    starts_all = np.arange(0, n - bl + 1)
    idx_of = {f: PREDICTORS.index(f) + 1 for f in PREDICTORS}  # +1 for const
    draws = {f: [] for f in PREDICTORS}
    for _ in range(n_boot):
        starts = rng.choice(starts_all, size=n_blocks, replace=True)
        idx = np.concatenate([np.arange(s, s + bl) for s in starts])[:n]
        try:
            b = np.linalg.lstsq(X[idx], y[idx], rcond=None)[0]
        except np.linalg.LinAlgError:
            continue
        for f in PREDICTORS:
            draws[f].append(b[idx_of[f]])
    return {f: (float(np.mean(v)), float(np.std(v, ddof=1))) for f, v in draws.items()}


if RUN_BOOTSTRAP:
    boot_recs = []
    for keys, grp in offs.groupby(GROUP_COLS, observed=True):
        t, logd, fs = load_group_delta_nrem(*keys)
        et = build_epoch_table(grp, t, logd, fs, EPOCH_DURATION, MIN_NREM_FRAC)
        df = et[PREDICTORS + ["mean_log_delta"]].dropna().copy()
        for c in df.columns:
            df[c] = zscore(df[c])
        if (df[PREDICTORS].std(ddof=0) == 0).any() or len(df) < MIN_EPOCHS:
            continue
        bb = block_bootstrap_betas(df, EPOCH_DURATION)
        boot_recs.append(dict(
            zip(GROUP_COLS, keys),
            beta_blas=bb["blas_area"][0], se_blas=bb["blas_area"][1],
            beta_clas_excl=bb["clas_excl_area"][0], se_clas_excl=bb["clas_excl_area"][1],
            beta_llas_excl=bb["llas_excl_area"][0], se_llas_excl=bb["llas_excl_area"][1],
        ))
    boot_df = pd.DataFrame(boot_recs)
    display(
        pool_tiers(boot_df).style.format({
            "pooled_std_beta": "{:+.4f}", "ci_lo": "{:+.4f}", "ci_hi": "{:+.4f}",
            "p": "{:.2e}", "i_squared": "{:.0f}%",
        }).set_caption("Block-bootstrap pooled coefficient by tier")
    )
else:
    print("RUN_BOOTSTRAP = False (skipped)")

## Collapsed amount (area) model: NREM secondary (Conservative CLAS set + LLAS-exclusive)

A two-tier companion to the three-tier model above, mirroring the Wake amount model
exactly: BLAS and CLAS-exclusive are folded into one Conservative (CLAS set) predictor
(`cons_area = blas_area + clas_excl_area`), fit jointly with LLAS-exclusive. Same
per-group HAC-OLS and DerSimonian-Laird pooling, exported with the same schema as
`wake_area_*` so the figure notebook renders it like the Wake amount panels. In Wake,
epoch-scale BLAS area is near-degenerate and has to be folded in; NREM's BLAS is
well-estimated on its own, so here the collapse is a framing choice, the whole
conservative set against the liberal-only increment.


In [ ]:
# Collapsed amount (area) model: NREM secondary, two estimable tiers.
# Fold BLAS + CLAS-exclusive into a single "Conservative (CLAS set)" predictor
# (cons_area) alongside LLAS-exclusive, the exact analog of the Wake amount model.
# Same per-group HAC-OLS + DerSimonian-Laird pooling and the same on-disk schema as
# wake_area_*, so added_value_figures can render it like the Wake amount panels.
# (In NREM, unlike Wake, BLAS is well-estimated on its own, so this collapse is a
# presentation choice, not a necessity; it isolates the whole conservative CLAS
# set vs the liberal-only increment.)
COLLAPSED = ["cons_area", "llas_excl_area"]
COLLAPSED_LABELS = {"cons_area": "Conservative (CLAS set)",
                    "llas_excl_area": "LLAS-exclusive"}
COLL_COLS = {"cons_area": ("beta_cons_area", "se_cons_area"),
             "llas_excl_area": ("beta_llas_excl_area", "se_llas_excl_area")}


def fit_collapsed_group(epoch_df, epoch_duration=EPOCH_DURATION, transform="zscore"):
    """Per-group collapsed fit. Returns (partial, marginal, semipartial) dicts keyed
    by the beta_/se_ column names, or None if too few epochs / a predictor is
    constant. 'partial' is the joint 2-predictor HAC-OLS; 'marginal' is each tier
    alone; 'semipartial' is the joint coefficient rescaled to the part correlation
    (keys suffixed '_sr'), whose square is the tier's incremental R2.

    transform='rank' refits the identical model on within-group ranks (the
    partial-Spearman analog). cons_area is summed *before* ranking, so the rank pass
    ranks the conservative total rather than adding two rank vectors. maxlags is
    derived from epoch_duration, so the HAC window stays ~HAC_TARGET_S at every
    epoch length in the sweep.
    """
    base = epoch_df[PREDICTORS + ["mean_log_delta"]].dropna().copy()
    if len(base) < MIN_EPOCHS:
        return None
    base["cons_area"] = base["blas_area"] + base["clas_excl_area"]
    z = prep_columns(base[COLLAPSED + ["mean_log_delta"]], transform)
    if (z[COLLAPSED].std(ddof=0) == 0).any():
        return None
    maxlags = max(1, int(np.ceil(HAC_TARGET_S / epoch_duration)))
    y = z["mean_log_delta"]
    full = sm.OLS(y, sm.add_constant(z[COLLAPSED])).fit(
        cov_type="HAC", cov_kwds={"maxlags": maxlags})
    scales = semipartial_scales(z, COLLAPSED)
    partial, marginal, semipartial = {}, {}, {}
    for p in COLLAPSED:
        b, s = COLL_COLS[p]
        partial[b], partial[s] = full.params[p], full.bse[p]
        semipartial[f"{b}_sr"] = full.params[p] * scales[p]
        semipartial[f"{s}_sr"] = full.bse[p] * scales[p]
        fm = sm.OLS(y, sm.add_constant(z[p])).fit(
            cov_type="HAC", cov_kwds={"maxlags": maxlags})
        marginal[b], marginal[s] = fm.params[p], fm.bse[p]
    return partial, marginal, semipartial


def run_collapsed(epoch_duration=EPOCH_DURATION, transform="zscore"):
    """Fit the collapsed model in every group at one epoch length / transform.
    Returns (joint + semipartial frame, marginal frame)."""
    p_recs, m_recs = [], []
    for keys, grp in offs.groupby(GROUP_COLS, observed=True):
        t, logd, fs = load_group_delta_nrem(*keys)
        et = build_epoch_table(grp, t, logd, fs, epoch_duration, MIN_NREM_FRAC)
        r = fit_collapsed_group(et, epoch_duration, transform)
        if r is None:
            continue
        base_rec = dict(zip(GROUP_COLS, keys))
        p_recs.append({**base_rec, **r[0], **r[2]})
        m_recs.append({**base_rec, **r[1]})
    return pd.DataFrame(p_recs), pd.DataFrame(m_recs)


coll_group_df, coll_marg_df = run_collapsed()


def pool_collapsed(gdf, suffix=""):
    """Random-effects pool each collapsed tier's (beta, se) into a tidy table with
    the same columns as pool_tiers (tier, pooled_std_beta, ci_lo, ci_hi, ...).

    suffix="_sr" pools the semipartial coefficients instead of the joint ones."""
    rows = []
    for p in COLLAPSED:
        b, s = COLL_COLS[p]
        b, s = b + suffix, s + suffix
        m = random_effects_meta(gdf[b], gdf[s] ** 2)
        rows.append(dict(tier=COLLAPSED_LABELS[p], pooled_std_beta=m["pooled"],
                         ci_lo=m["ci_lo"], ci_hi=m["ci_hi"], p=m["p"],
                         i_squared=m["i_squared"], k=m["k"]))
    return pd.DataFrame(rows)


nrem_area_partial_pooled = pool_collapsed(coll_group_df)
nrem_area_semipartial_pooled = pool_collapsed(coll_group_df, suffix="_sr")
nrem_area_marg_pooled = pool_collapsed(coll_marg_df)
for _, r in nrem_area_partial_pooled.iterrows():
    print(f"partial  {r['tier']:26s}: amount beta = {r['pooled_std_beta']:+.4f} "
          f"[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]  k = {int(r['k'])}")
for _, r in nrem_area_semipartial_pooled.iterrows():
    print(f"semipart {r['tier']:26s}: amount beta = {r['pooled_std_beta']:+.4f} "
          f"[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]  dR2 = {r['pooled_std_beta']**2:.4f}")
for _, r in nrem_area_marg_pooled.iterrows():
    print(f"marginal {r['tier']:26s}: amount beta = {r['pooled_std_beta']:+.4f} "
          f"[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]")

## Robustness: rank sibling and epoch sweep on the collapsed model

The rank-transform sibling and epoch sweep above are run on the three-tier model.
Supplementary Table S2b reports the two-tier collapsed model, so both checks are
repeated on it directly here, pooling the semipartial, the quantity S2b actually
tabulates. Without this the robustness claim in the Methods would be about a different
model than the one in the table.


In [ ]:
# Collapsed-model robustness: rank sibling x epoch-length sweep, run on the same
# two-tier model Table S2b reports and pooling the SEMIPARTIAL (the tabulated
# quantity). fit_collapsed_group derives maxlags from the epoch length, so the HAC
# window stays ~5 min across the sweep.
coll_rob_rows = []
for ep in EPOCH_SWEEP:
    for method, tr in [("OLS", "zscore"), ("rank", "rank")]:
        gd, _ = run_collapsed(ep, tr)
        if not len(gd):
            print(f"  skip {ep:.0f}s / {method}: no group fits")
            continue
        pt = pool_collapsed(gd, suffix="_sr")
        pt.insert(0, "method", method)
        pt.insert(0, "epoch_s", ep)
        coll_rob_rows.append(pt)
coll_rob_df = pd.concat(coll_rob_rows, ignore_index=True)

for _, r in coll_rob_df.iterrows():
    print(f"{r['epoch_s']:5.0f}s {r['method']:4s} {r['tier']:26s}: "
          f"semipartial = {r['pooled_std_beta']:+.4f} "
          f"[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]  p = {r['p']:.2e}  "
          f"I2 = {r['i_squared']:.0f}%  k = {int(r['k'])}")

display(
    methods_wide(coll_rob_df, ["epoch_s", "tier"]).style.set_caption(
        "Collapsed amount model: pooled SEMIPARTIAL coefficient by tier, epoch "
        "length and transform (the quantity reported in Supplementary Table S2b)"
    )
)

## Interpretation

Reading the headline, the pooled `llas_excl_area` coefficient:

- CI excludes 0, either sign: the liberal-only OFFs carry unique variance not captured
  by BLAS/CLAS, so they add value. The sign gives the direction of the conditional
  association. A significant negative coefficient is still added value, the events are
  informative, just pointing opposite to the conservative tiers (e.g. epochs dominated
  by small/fragmented OFFs rather than big consolidated ones).
- CI brackets 0 under the conservative HAC + random-effects inference: the liberal-only
  OFFs are redundant with the conservative OFFs at this timescale. They may still have
  value alone without adding it, which is the distinction the marginal-correlation
  notebooks could not make.

The levels model (cross-epoch, between-state) and the first-difference model
(moment-to-moment fluctuations) answer different questions and can disagree in sign.
Report both rather than collapsing them into one number; with low VIF a sign flip is a
finding about timescale and not a collinearity artifact.

Limitations:

- Within-subject clustering. `(subject, probe, structure)` groups within a subject are
  not independent; the two-stage RE meta-analysis treats them as exchangeable. The
  fully-correct version is a 3-level mixed model with an AR(1) residual, which belongs
  in `r-offp` downstream.
- HAC across NREM gaps. Epochs are contiguous in NREM time but Wake/REM removal
  introduces gaps, so lag-k is not exactly k*epoch in wall-clock time. The
  block-bootstrap and first-difference checks are robust to this.
- Source. Full-48h morphological OFFs are the current best-but-provisional source.
- Predictor. Total OFF area is the primary summary; OFF occupancy / total OFF-time and
  count/rate are reasonable alternatives (swap `area` in `build_epoch_table`).

If this analysis is adopted, the kernels (`build_epoch_table`, `fit_group`,
`random_effects_meta`) should move into a module.

The rank-transform sibling is reported next to the OLS in the summary, epoch-sweep and
first-difference tables, along with a per-group OLS-vs-rank agreement scatter. Agreement
there indicates the zero-inflation and heavy-tail concern is not driving the result. The
three-tier checks do not license a claim about the collapsed two-tier model that
Supplementary Table S2b reports, so the rank sibling and the epoch sweep are run on the
collapsed model directly, pooling the semipartial coefficient S2b tabulates.

The marginal-vs-partial table makes the original question explicit: a positive marginal
association with a negative partial coefficient, as expected for LLAS-exclusive, means
those OFFs have delta-relevant signal but do not add it in the same direction once the
bigger OFFs are held fixed.


## Export figure data

Write the quantities the figure notebook re-consumes, so it does not recompute them.


In [ ]:
import pathlib

DATA = pathlib.Path("./outputs/added_value_data")
DATA.mkdir(parents=True, exist_ok=True)
meta_summary.to_parquet(DATA / "nrem_partial_pooled.parquet")
meta_semipartial.to_parquet(DATA / "nrem_semipartial_pooled.parquet")
meta_marg.to_parquet(DATA / "nrem_marginal_pooled.parquet")
group_df[GROUP_COLS + ["beta_blas", "se_blas", "beta_clas_excl", "se_clas_excl",
                       "beta_llas_excl", "se_llas_excl",
                       "beta_blas_sr", "se_blas_sr", "beta_clas_excl_sr",
                       "se_clas_excl_sr", "beta_llas_excl_sr", "se_llas_excl_sr"]].to_parquet(
    DATA / "nrem_group_partial.parquet")
pooled[["blas_area", "clas_excl_area", "llas_excl_area", "mean_log_delta"]].to_parquet(
    DATA / "nrem_strat_epochs.parquet")
# Collapsed amount (area) model: NREM secondary; two estimable tiers
# (Conservative = BLAS + CLAS-exclusive, and LLAS-exclusive). Same schema as wake_area_*.
nrem_area_partial_pooled.to_parquet(DATA / "nrem_area_partial_pooled.parquet")
nrem_area_semipartial_pooled.to_parquet(DATA / "nrem_area_semipartial_pooled.parquet")
nrem_area_marg_pooled.to_parquet(DATA / "nrem_area_marginal_pooled.parquet")
coll_group_df[GROUP_COLS + ["beta_cons_area", "se_cons_area",
                            "beta_llas_excl_area", "se_llas_excl_area",
                            "beta_cons_area_sr", "se_cons_area_sr",
                            "beta_llas_excl_area_sr", "se_llas_excl_area_sr"]].to_parquet(
    DATA / "nrem_area_group_partial.parquet")
# Collapsed-model robustness grid (rank sibling x epoch sweep, semipartial) -- the
# evidence behind the robustness sentence in the manuscript Methods for Table S2b.
coll_rob_df.to_parquet(DATA / "nrem_area_robustness.parquet")
print("exported NREM figure data ->", DATA.resolve())